# Normalización de `Cars_Datasets_2025.csv` (UNF → 1FN → 2FN → 3FN → 4FN)



- El paquete `pyodbc` y `sqlalchemy` instalados (`pip install pyodbc sqlalchemy`)
- El **ODBC Driver 17 o 18 para SQL Server** instalado en tu máquina
- Una instancia de SQL Server accesible (local, en red, o Azure SQL) con permisos para crear tablas

In [1]:
import pandas as pd
import re
import os

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 160)

## 0. Datos sin normalizar (UNF)

Cargamos el CSV tal como viene.

In [2]:
df = pd.read_csv('Cars_Datasets_2025.csv', encoding='cp1252')
df = df.reset_index(drop=False).rename(columns={'index': 'row_id'})
print(df.shape)
df.head(5)

(1218, 12)


,row_id,Company Names,Cars Names,Engines,CC/Battery Capacity,HorsePower,Total Speed,Performance(0 - 100 )KM/H,Cars Prices,Fuel Types,Seats,Torque
0,0,FERRARI,SF90 STRADALE,V8,3990 cc,963 hp,340 km/h,2.5 sec,"$1,100,000",plug in hyrbrid,2,800 Nm
1,1,ROLLS ROYCE,PHANTOM,V12,6749 cc,563 hp,250 km/h,5.3 sec,"$460,000",Petrol,5,900 Nm
2,2,Ford,KA+,1.2L Petrol,"1,200 cc",70-85 hp,165 km/h,10.5 sec,"$12,000-$15,000",Petrol,5,100 - 140 Nm
3,3,MERCEDES,GT 63 S,V8,"3,982 cc",630 hp,250 km/h,3.2 sec,"$161,000",Petrol,4,900 Nm
4,4,AUDI,AUDI R8 Gt,V10,"5,204 cc",602 hp,320 km/h,3.6 sec,"$253,290",Petrol,2,560 Nm


In [3]:
# Diagnóstico de atomicidad: ¿qué columnas NO son atómicas?
ejemplo_touareg = df[df['Cars Names'] == 'Touareg'].iloc[0]
print(ejemplo_touareg[['Company Names','Cars Names','Engines','CC/Battery Capacity','HorsePower',
                        'Total Speed','Performance(0 - 100 )KM/H','Cars Prices','Fuel Types','Seats','Torque']])

Company Names                                    Volkswagen
Cars Names                                          Touareg
Engines                      3.0L V6 Turbo Diesel / 4.0L V8
CC/Battery Capacity                       2967 cc / 3956 cc
HorsePower                                     231 - 500 hp
Total Speed                                        235 km/h
Performance(0 - 100 )KM/H                           5.9 sec
Cars Prices                                        $55,000 
Fuel Types                                   Petrol, Diesel
Seats                                                     5
Torque                                               600 Nm
Name: 279, dtype: object


Este único renglón (Volkswagen Touareg) esconde varios problemas de atomicidad:

| Atributo | Problema |
|---|---|
| `CC/Battery Capacity` | Compuesto (número + unidad) **y** lista: `"2967 cc / 3956 cc"` |
| `HorsePower` | Compuesto y a veces rango: `"231 - 500 hp"` |
| `Total Speed`, `Performance` | Compuesto: número pegado a la unidad |
| `Cars Prices` | Compuesto, a veces rango |
| `Fuel Types` | **Multivaluado**: `"Petrol, Diesel"` |
| `Seats` | A veces notación especial: `"2+2"` |
| `Torque` | Compuesto, a veces rango |
| `Company Names` | Atómico pero sucio: `"KIA"`, `"KIA  "`, `"Kia"` son la misma marca |

Dos de estas columnas (`Fuel Types` y `CC/Battery Capacity`) son listas **independientes** dentro de la misma celda — eso es justo lo que la 4FN va a resolver.

## 1. Primera Forma Normal (1FN)

Reglas: todo atributo debe ser atómico, sin rangos ni listas en una sola celda.
	`¿Esta celda tiene más de un valor?`

Primero, limpiamos el nombre de la empresa (37 valores crudos → nombres reales de marca).

In [4]:
company_map = {
    'ASTON MARTIN':'Aston Martin','AUDI':'Audi','Acura':'Acura','BENTLEY':'Bentley','BMW':'BMW',
    'Bugatti':'Bugatti','Cadillac':'Cadillac','Chevrolet':'Chevrolet','FERRARI':'Ferrari','Ford':'Ford',
    'GMC':'GMC','HONDA':'Honda','HYUNDAI':'Hyundai','Jaguar Land Rover':'Jaguar Land Rover','Jeep':'Jeep',
    'KIA':'Kia','KIA  ':'Kia','Kia':'Kia','LAMBORGHINI':'Lamborghini','MAHINDRA':'Mahindra',
    'MARUTI SUZUKI':'Maruti Suzuki','MERCEDES':'Mercedes-Benz','Mazda':'Mazda','Mitsubishi':'Mitsubishi',
    'NISSAN':'Nissan','Nissan':'Nissan','Peugeot':'Peugeot','Porsche':'Porsche','ROLLS ROYCE':'Rolls Royce',
    'ROLLS ROYCE ':'Rolls Royce','TOYOTA':'Toyota','Tata Motors':'Tata Motors','Tesla':'Tesla','Toyota':'Toyota',
    'VOLVO':'Volvo','Volkswagen':'Volkswagen','Volvo':'Volvo',
}
df['empresa'] = df['Company Names'].map(company_map)
assert df['empresa'].isna().sum() == 0
print(df['Company Names'].nunique(), '->', df['empresa'].nunique(), 'marcas')

37 -> 31 marcas


In [5]:
# --- Funciones para atomizar valores compuestos y rangos ---

def nums(s):
    """Extrae todos los números de un texto, p.ej. '231 - 500 hp' -> [231.0, 500.0]"""
    if pd.isna(s):
        return []
    return [float(x.replace(',', '')) for x in re.findall(r'[\d,]*\d(?:\.\d+)?', str(s))]

def minmax(s):
    """Devuelve (min, max). Si el valor es único, min == max."""
    ns = nums(s)
    if not ns:
        return (None, None)
    return (min(ns), max(ns))

def seats_minmax(s):
    """'2+2' = 2 principales + 2 ocasionales (total 4). '2-12' = rango normal."""
    s = str(s).strip()
    if s in ('nan', ''):
        return (None, None)
    if '+' in s:
        a, b = s.split('+')
        a, b = float(a), float(b)
        return (a, a + b)
    ns = nums(s)
    return (min(ns), max(ns)) if ns else (None, None)

df['hp_min'], df['hp_max'] = zip(*df['HorsePower'].map(minmax))
df['vel_max_kmh'] = df['Total Speed'].map(lambda s: nums(s)[0] if nums(s) else None)
df['acel_0_100_seg'] = df['Performance(0 - 100 )KM/H'].map(lambda s: nums(s)[0] if nums(s) else None)
df['precio_min_usd'], df['precio_max_usd'] = zip(*df['Cars Prices'].map(minmax))
df['torque_min_nm'], df['torque_max_nm'] = zip(*df['Torque'].map(minmax))
df['asientos_min'], df['asientos_max'] = zip(*df['Seats'].map(seats_minmax))

df[['Cars Names','HorsePower','hp_min','hp_max','Seats','asientos_min','asientos_max']].head(5)

,Cars Names,HorsePower,hp_min,hp_max,Seats,asientos_min,asientos_max
0,SF90 STRADALE,963 hp,963.0,963.0,2,2.0,2.0
1,PHANTOM,563 hp,563.0,563.0,5,5.0,5.0
2,KA+,70-85 hp,70.0,85.0,5,5.0,5.0
3,GT 63 S,630 hp,630.0,630.0,4,4.0,4.0
4,AUDI R8 Gt,602 hp,602.0,602.0,2,2.0,2.0


In [6]:
# --- Explotar los dos grupos repetitivos: combustibles y opciones de motor/batería ---

fuel_fix = {
    'plug in hyrbrid':'Plug-in Hybrid', 'plug-in hybrid':'Plug-in Hybrid', 'hybrid (petrol)':'Hybrid',
    'petrol (hybrid)':'Hybrid', 'hybrid/petrol':'Hybrid', 'diesel hybrid':'Hybrid',
    'hybrid (gas + electric)':'Hybrid', 'gas':'Petrol', 'ev':'Electric', 'awd':'AWD', 'cng':'CNG',
}

def split_fuel(s):
    """'Petrol, Diesel' -> ['Diesel', 'Petrol'] (multivaluado, separado por ',' o '/')"""
    parts = re.split(r'[,/]', str(s))
    out = []
    for p in parts:
        p = p.strip().lower()
        p = fuel_fix.get(p, p)
        p = p if p in ('AWD', 'CNG') else p.capitalize()
        out.append(p)
    return sorted(set(out))

def split_capacity(s):
    """'2967 cc / 3956 cc' -> [(2967.0,'cc'), (3956.0,'cc')]. Detecta unidad cc o kWh por segmento."""
    if pd.isna(s):
        return []
    out = []
    for seg in re.split(r'[,/]', str(s)):
        low = seg.lower()
        unidad = 'kWh' if 'kwh' in low else ('cc' if 'cc' in low else None)
        if unidad is None:
            continue
        out.extend((v, unidad) for v in nums(seg))
    seen, res = set(), []
    for t in out:
        if t not in seen:
            seen.add(t); res.append(t)
    return res

df['combustibles'] = df['Fuel Types'].map(split_fuel)
df['opciones_motor'] = df['CC/Battery Capacity'].map(split_capacity)

print('Autos con más de un combustible:', (df['combustibles'].map(len) > 1).sum())
print('Autos con más de una opción de motor/batería:', (df['opciones_motor'].map(len) > 1).sum())

Autos con más de un combustible: 49
Autos con más de una opción de motor/batería: 79


In [7]:
# Reconstrucción de la tabla 1FN completamente plana para el Touareg (fan-out 2x2 = 4 filas)
t = df[df['Cars Names'] == 'Touareg'].iloc[0]
filas_1fn = [
    {'Empresa': t['empresa'], 'Modelo': t['Cars Names'], 'Combustible': c,
     'Motor_valor': v, 'Motor_unidad': u, 'HP_min': t['hp_min'], 'Precio_min': t['precio_min_usd']}
    for c in t['combustibles'] for v, u in t['opciones_motor']
]
pd.DataFrame(filas_1fn)

,Empresa,Modelo,Combustible,Motor_valor,Motor_unidad,HP_min,Precio_min
0,Volkswagen,Touareg,Diesel,2967.0,cc,231.0,55000.0
1,Volkswagen,Touareg,Diesel,3956.0,cc,231.0,55000.0
2,Volkswagen,Touareg,Petrol,2967.0,cc,231.0,55000.0
3,Volkswagen,Touareg,Petrol,3956.0,cc,231.0,55000.0


Un solo auto pasó a ocupar **4 filas**, con el precio y el HP repetidos idénticos en las 4. Además, descubrimos que **(Empresa, Modelo) no es una clave candidata válida**: hay 16 combinaciones repetidas con especificaciones distintas (variantes/versiones cargadas por separado). Por eso, de aquí en adelante usamos un identificador sustituto: `modelo_id`.

## 2. Segunda Forma Normal (2FN)

Qué exige: que todo dato dependa de la clave completa, no solo de una parte de ella. (Esto solo importa cuando la clave está compuesta por más de una columna.)
`¿Este dato depende de TODA la clave, o solo de una parte?`
`HP_min`, `Precio_min`, `Vel_max`, etc. dependen solo de (Empresa, Modelo) — **no** del combustible ni de la opción de motor. Esa es una dependencia parcial. Se separan en tres tablas, ya con `modelo_id` como clave.

In [8]:
modelo = df[['row_id','empresa','Cars Names','Engines','vel_max_kmh','acel_0_100_seg',
             'asientos_min','asientos_max','precio_min_usd','precio_max_usd',
             'hp_min','hp_max','torque_min_nm','torque_max_nm']].copy()
modelo.columns = ['modelo_id','empresa','nombre_modelo','motor_config','vel_max_kmh','acel_0_100_seg',
                   'asientos_min','asientos_max','precio_min_usd','precio_max_usd',
                   'hp_min','hp_max','torque_min_nm','torque_max_nm']
modelo['modelo_id'] += 1  # IDs 1..1218

modelo_combustible = pd.DataFrame(
    [(r.row_id + 1, c) for r in df.itertuples() for c in r.combustibles],
    columns=['modelo_id', 'combustible']
).drop_duplicates()

modelo_opcion_motor = pd.DataFrame(
    [(r.row_id + 1, v, u) for r in df.itertuples() for v, u in r.opciones_motor],
    columns=['modelo_id', 'valor', 'unidad']
).drop_duplicates()

print('modelo:', len(modelo), '| modelo_combustible:', len(modelo_combustible), '| modelo_opcion_motor:', len(modelo_opcion_motor))
modelo.head(3)

modelo: 1218 | modelo_combustible: 1267 | modelo_opcion_motor: 1311


,modelo_id,empresa,nombre_modelo,motor_config,vel_max_kmh,acel_0_100_seg,asientos_min,asientos_max,precio_min_usd,precio_max_usd,hp_min,hp_max,torque_min_nm,torque_max_nm
0,1,Ferrari,SF90 STRADALE,V8,340.0,2.5,2.0,2.0,1100000.0,1100000.0,963.0,963.0,800.0,800.0
1,2,Rolls Royce,PHANTOM,V12,250.0,5.3,5.0,5.0,460000.0,460000.0,563.0,563.0,900.0,900.0
2,3,Ford,KA+,1.2L Petrol,165.0,10.5,5.0,5.0,12000.0,15000.0,70.0,85.0,100.0,140.0


## 3. Tercera Forma Normal (3FN)
Qué exige: que ningún dato dependa de otro dato que no es la clave. Es decir, no debe haber una "cadena": Clave → Dato_A → Dato_B.
`¿Este dato depende de la clave, o de otro dato de la fila?`
`modelo_id → empresa → país → continente` es una dependencia transitiva: el continente depende del país, no directamente del modelo. Si dejáramos "Volkswagen"/"Alemania" repetidos en cada uno de sus modelos, cualquier corrección habría que hacerla decenas de veces.

*Nota: País y Continente de cada marca son datos reales de dominio público, agregados aquí para poder ilustrar este paso — no vienen en el CSV.*

In [9]:
pais_map = {
    'Aston Martin':'Reino Unido','Audi':'Alemania','Acura':'Japón','Bentley':'Reino Unido','BMW':'Alemania',
    'Bugatti':'Francia','Cadillac':'Estados Unidos','Chevrolet':'Estados Unidos','Ferrari':'Italia','Ford':'Estados Unidos',
    'GMC':'Estados Unidos','Honda':'Japón','Hyundai':'Corea del Sur','Jaguar Land Rover':'Reino Unido','Jeep':'Estados Unidos',
    'Kia':'Corea del Sur','Lamborghini':'Italia','Mahindra':'India','Maruti Suzuki':'India','Mercedes-Benz':'Alemania',
    'Mazda':'Japón','Mitsubishi':'Japón','Nissan':'Japón','Peugeot':'Francia','Porsche':'Alemania','Rolls Royce':'Reino Unido',
    'Toyota':'Japón','Tata Motors':'India','Tesla':'Estados Unidos','Volvo':'Suecia','Volkswagen':'Alemania',
}
continente_map = {
    'Reino Unido':'Europa','Alemania':'Europa','Francia':'Europa','Italia':'Europa','Suecia':'Europa',
    'Estados Unidos':'América del Norte','Japón':'Asia','Corea del Sur':'Asia','India':'Asia',
}

empresas = sorted(df['empresa'].unique())
paises = sorted(set(pais_map.values()))
continentes = sorted(set(continente_map.values()))
emp_id = {e: i + 1 for i, e in enumerate(empresas)}

tabla_continente = pd.DataFrame({'continente': continentes})
tabla_pais = pd.DataFrame([(p, continente_map[p]) for p in paises], columns=['pais', 'continente'])
tabla_empresa = pd.DataFrame([(emp_id[e], e, pais_map[e]) for e in empresas],
                              columns=['empresa_id', 'nombre', 'pais'])

# modelo ya no guarda el nombre de la empresa como texto: solo empresa_id
modelo['empresa_id'] = modelo['empresa'].map(emp_id)
modelo = modelo.drop(columns=['empresa'])

print(len(tabla_continente), 'continentes |', len(tabla_pais), 'países |', len(tabla_empresa), 'empresas')
tabla_empresa.head(5)

3 continentes | 9 países | 31 empresas


,empresa_id,nombre,pais
0,1,Acura,Japón
1,2,Aston Martin,Reino Unido
2,3,Audi,Alemania
3,4,BMW,Alemania
4,5,Bentley,Reino Unido


## 4. Cuarta Forma Normal (4FN)
Qué exige: que no se mezclen en una misma tabla dos listas independientes de la misma entidad.
`¿Estoy mezclando dos listas independientes en una tabla?`
`modelo_id →→ combustible` y `modelo_id →→ opción_motor` son dos dependencias multivaluadas **independientes**: qué combustibles acepta el auto y qué variantes de motor/batería se ofrecen son hechos distintos. Guardarlas juntas obliga a multiplicar filas (viste el fan-out 2×2 del Touareg en la sección 1). Como ya las separamos desde la 2FN por necesidad práctica, **el esquema ya cumple 4FN**.



### Esquema final

```
continente          (continente PK)
pais                (pais PK, continente FK)
empresa             (empresa_id PK, nombre, pais FK)
modelo              (modelo_id PK, empresa_id FK, nombre_modelo, motor_config,
                      vel_max_kmh, acel_0_100_seg, asientos_min/max,
                      precio_min/max_usd, hp_min/max, torque_min/max_nm)
modelo_combustible  (modelo_id FK, combustible)          PK: ambos
modelo_opcion_motor (modelo_id FK, valor, unidad)         PK: los tres
```

In [23]:
# Verificación: reconstruir TODOS los "Touareg" desde las tablas normalizadas (JOIN)
ids = modelo.loc[modelo['nombre_modelo'] == 'Touareg', 'modelo_id']
print("IDs encontrados:", ids.tolist())

recon = (modelo_combustible[modelo_combustible['modelo_id'].isin(ids)]
         .merge(modelo_opcion_motor[modelo_opcion_motor['modelo_id'].isin(ids)],
                on='modelo_id')
         .merge(modelo[['modelo_id', 'nombre_modelo']], on='modelo_id')
         .sort_values(['modelo_id', 'combustible', 'valor'])
         .reset_index(drop=True))
recon

IDs encontrados: [280]


,modelo_id,combustible,valor,unidad,nombre_modelo
0,280,Diesel,2967.0,cc,Touareg
1,280,Diesel,3956.0,cc,Touareg
2,280,Petrol,2967.0,cc,Touareg
3,280,Petrol,3956.0,cc,Touareg


## Conectarse a SQL Server

Ajusta estos datos de conexión según tu instancia. Puedes usar autenticación de Windows (`Trusted_Connection=yes`) o autenticación de SQL Server (usuario/contraseña) — deja solo una de las dos opciones activa.

In [12]:
import pyodbc
from sqlalchemy import create_engine, text

SERVER   = "localhost\SQLEXPRESS"          # p.ej. "localhost\\SQLEXPRESS" o "mi-servidor.database.windows.net"
DATABASE = "carritos_4fn"           # la base debe existir de antemano (créala en SSMS: CREATE DATABASE cars_4fn;)
DRIVER   = "ODBC Driver 17 for SQL Server"

# --- Opción A: autenticación de Windows ---
#conn_str = (
#    f"DRIVER={{{DRIVER}}};SERVER={SERVER};DATABASE={DATABASE};"
#    f"Trusted_Connection=yes;"
#)

# --- Opción B: autenticación de SQL Server (usuario/contraseña) ---
USER = "sa"
PASSWORD = "******"
conn_str = (
     f"DRIVER={{{DRIVER}}};SERVER={SERVER};DATABASE={DATABASE};"
     f"UID={USER};PWD={PASSWORD};"
 )

engine = create_engine(f"mssql+pyodbc:///?odbc_connect={conn_str}", fast_executemany=True)

with engine.connect() as conn:
    version = conn.execute(text("SELECT @@VERSION")).scalar()
print(version)

<>:4: SyntaxWarning: "\S" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\S"? A raw string is also an option.
<>:4: SyntaxWarning: "\S" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\S"? A raw string is also an option.
C:\Users\cadlj\AppData\Local\Temp\ipykernel_14900\1444173367.py:4: SyntaxWarning: "\S" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\S"? A raw string is also an option.
  SERVER   = "localhost\SQLEXPRESS"          # p.ej. "localhost\\SQLEXPRESS" o "mi-servidor.database.windows.net"


Microsoft SQL Server 2025 (RTM) - 17.0.1000.7 (X64) 
	Oct 21 2025 12:05:57 
	Copyright (C) 2025 Microsoft Corporation
	Express Edition (64-bit) on Windows 10 Home Single Language 10.0 <X64> (Build 26200: ) (Hypervisor)



## Crear las tablas (4FN) en SQL Server

In [13]:
ddl = """
IF OBJECT_ID('dbo.modelo_opcion_motor','U') IS NOT NULL DROP TABLE dbo.modelo_opcion_motor;
IF OBJECT_ID('dbo.modelo_combustible','U')  IS NOT NULL DROP TABLE dbo.modelo_combustible;
IF OBJECT_ID('dbo.modelo','U')              IS NOT NULL DROP TABLE dbo.modelo;
IF OBJECT_ID('dbo.empresa','U')             IS NOT NULL DROP TABLE dbo.empresa;
IF OBJECT_ID('dbo.pais','U')                IS NOT NULL DROP TABLE dbo.pais;
IF OBJECT_ID('dbo.continente','U')          IS NOT NULL DROP TABLE dbo.continente;

CREATE TABLE dbo.continente (
    continente NVARCHAR(30) NOT NULL,
    CONSTRAINT PK_continente PRIMARY KEY (continente)
);

CREATE TABLE dbo.pais (
    pais       NVARCHAR(30) NOT NULL,
    continente NVARCHAR(30) NOT NULL,
    CONSTRAINT PK_pais PRIMARY KEY (pais),
    CONSTRAINT FK_pais_continente FOREIGN KEY (continente) REFERENCES dbo.continente (continente)
);

CREATE TABLE dbo.empresa (
    empresa_id INT NOT NULL,
    nombre     NVARCHAR(50) NOT NULL,
    pais       NVARCHAR(30) NOT NULL,
    CONSTRAINT PK_empresa PRIMARY KEY (empresa_id),
    CONSTRAINT UQ_empresa_nombre UNIQUE (nombre),
    CONSTRAINT FK_empresa_pais FOREIGN KEY (pais) REFERENCES dbo.pais (pais)
);

CREATE TABLE dbo.modelo (
    modelo_id      INT NOT NULL,
    empresa_id     INT NOT NULL,
    nombre_modelo  NVARCHAR(100) NOT NULL,
    motor_config   NVARCHAR(100),
    vel_max_kmh    FLOAT,
    acel_0_100_seg FLOAT,
    asientos_min   FLOAT,
    asientos_max   FLOAT,
    precio_min_usd FLOAT,
    precio_max_usd FLOAT,
    hp_min         FLOAT,
    hp_max         FLOAT,
    torque_min_nm  FLOAT,
    torque_max_nm  FLOAT,
    CONSTRAINT PK_modelo PRIMARY KEY (modelo_id),
    CONSTRAINT FK_modelo_empresa FOREIGN KEY (empresa_id) REFERENCES dbo.empresa (empresa_id)
);

CREATE TABLE dbo.modelo_combustible (
    modelo_id   INT NOT NULL,
    combustible NVARCHAR(30) NOT NULL,
    CONSTRAINT PK_modelo_combustible PRIMARY KEY (modelo_id, combustible),
    CONSTRAINT FK_mc_modelo FOREIGN KEY (modelo_id) REFERENCES dbo.modelo (modelo_id)
);

CREATE TABLE dbo.modelo_opcion_motor (
    modelo_id INT NOT NULL,
    valor     FLOAT NOT NULL,
    unidad    NVARCHAR(10) NOT NULL,
    CONSTRAINT PK_modelo_opcion_motor PRIMARY KEY (modelo_id, valor, unidad),
    CONSTRAINT FK_mo_modelo FOREIGN KEY (modelo_id) REFERENCES dbo.modelo (modelo_id)
);
"""

with engine.begin() as conn:
    for statement in ddl.split(";"):
        if statement.strip():
            conn.execute(text(statement))
print("Tablas creadas.")

Tablas creadas.


## Cargar los DataFrames a SQL Server

In [14]:
tabla_continente.to_sql('continente', engine, schema='dbo', if_exists='append', index=False)
tabla_pais.to_sql('pais', engine, schema='dbo', if_exists='append', index=False)
tabla_empresa.to_sql('empresa', engine, schema='dbo', if_exists='append', index=False)
modelo.to_sql('modelo', engine, schema='dbo', if_exists='append', index=False)
modelo_combustible.to_sql('modelo_combustible', engine, schema='dbo', if_exists='append', index=False)
modelo_opcion_motor.to_sql('modelo_opcion_motor', engine, schema='dbo', if_exists='append', index=False)
print("Datos cargados.")

Datos cargados.


In [15]:
with engine.connect() as conn:
    for t in ['continente','pais','empresa','modelo','modelo_combustible','modelo_opcion_motor']:
        n = conn.execute(text(f"SELECT COUNT(*) FROM dbo.{t}")).scalar()
        print(t, n, 'filas')
    # Esperado: 3, 9, 31, 1218, 1267, 1311

continente 3 filas
pais 9 filas
empresa 31 filas
modelo 1218 filas
modelo_combustible 1267 filas
modelo_opcion_motor 1311 filas


## Consultas de ejemplo sobre el esquema ya normalizado

In [27]:
from IPython.display import display

# Datos de TODOS los "Touareg", SIN usar JOIN: una consulta por tabla, relacionadas con subconsultas (IN)
datos = pd.read_sql(text("""
    SELECT modelo_id, nombre_modelo, motor_config, hp_min, hp_max, precio_min_usd, precio_max_usd
    FROM dbo.modelo
    WHERE nombre_modelo = 'Touareg'
"""), engine)
print("Modelo:")
display(datos)

combustibles = pd.read_sql(text("""
    SELECT modelo_id, combustible
    FROM dbo.modelo_combustible
    WHERE modelo_id IN (SELECT modelo_id FROM dbo.modelo WHERE nombre_modelo = 'Touareg')
    ORDER BY modelo_id, combustible
"""), engine)
print("Combustibles:")
display(combustibles)

motores = pd.read_sql(text("""
    SELECT modelo_id, valor, unidad
    FROM dbo.modelo_opcion_motor
    WHERE modelo_id IN (SELECT modelo_id FROM dbo.modelo WHERE nombre_modelo = 'Touareg')
    ORDER BY modelo_id, valor
"""), engine)
print("Opciones de motor:")
display(motores)

empresa_pais = pd.read_sql(text("""
    SELECT DISTINCT nombre AS empresa, pais
    FROM dbo.empresa
    WHERE empresa_id IN (SELECT empresa_id FROM dbo.modelo WHERE nombre_modelo = 'Touareg')
"""), engine)
print("Empresa y país:")
display(empresa_pais)

Modelo:


,modelo_id,nombre_modelo,motor_config,hp_min,hp_max,precio_min_usd,precio_max_usd
0,280,Touareg,3.0L V6 Turbo Diesel / 4.0L V8,231.0,500.0,55000.0,55000.0


Combustibles:


,modelo_id,combustible
0,280,Diesel
1,280,Petrol


Opciones de motor:


,modelo_id,valor,unidad
0,280,2967.0,cc
1,280,3956.0,cc


Empresa y país:


,empresa,pais
0,Volkswagen,Alemania


In [17]:
# Precio promedio por país, solo autos con más de 500 hp (con subconsulta, sin JOIN)
# SQL Server no permite una subconsulta directamente en GROUP BY: primero se resuelve
# el país en una subconsulta dentro del FROM, y se agrupa por esa columna ya calculada.
pd.read_sql(text("""
    SELECT pais, COUNT(*) AS modelos, ROUND(AVG(precio_min_usd), 0) AS precio_promedio
    FROM (
        SELECT
            precio_min_usd,
            (SELECT pais FROM dbo.empresa WHERE empresa_id = modelo.empresa_id) AS pais
        FROM dbo.modelo
        WHERE hp_min > 500
    ) AS t
    GROUP BY pais
    ORDER BY precio_promedio DESC
"""), engine)

,pais,modelos,precio_promedio
0,Francia,10,5870000.0
1,Japón,12,709333.0
2,Italia,33,612485.0
3,Reino Unido,49,428784.0
4,Alemania,53,178064.0
5,Suecia,2,140000.0
6,Estados Unidos,21,121453.0
7,Corea del Sur,1,65000.0


## Resumen

| Forma | Problema que resolvió | Cambio principal |
|---|---|---|
| **1FN** | Compuestos, rangos y listas en una celda | Se separaron min/max, unidades, y una fila por combustible y por opción de motor |
| **2FN** | (Empresa, Modelo) no era clave válida + dependencias parciales | Se creó `modelo_id` y se separaron `modelo`, `modelo_combustible`, `modelo_opcion_motor` |
| **3FN** | Dependencia transitiva modelo→empresa→país→continente | Se crearon `empresa`, `pais`, `continente` |
| **4FN** | Dos listas independientes (combustible y motor) mezcladas | Ya estaban separadas desde la 2FN por necesidad |

